## 0. Configuration et imports

In [ ]:
import os, sys, json, time
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from dotenv import load_dotenv, find_dotenv
sys.path.insert(0, os.path.abspath('..'))
load_dotenv(find_dotenv(), override=True)

In [ ]:
from opensearchpy import OpenSearch
import pandas as pd
import os

# Client OpenSearch
os_client = OpenSearch(
    hosts=[{"host": os.getenv("OPENSEARCH_HOST", "localhost"),
            "port": int(os.getenv("OPENSEARCH_PORT", 9200))}],
    http_auth=(
        os.getenv("OPENSEARCH_USER", "admin"),
        os.getenv("OPENSEARCH_PASSWORD", "R@gTime2026#Store")
    ),
    use_ssl=True,
    verify_certs=False,
    ssl_show_warn=False,
)
info = os_client.info()
print(f"OpenSearch {info['version']['number']} connecté")

# Chargement dataset prétraité
PREPROCESSED_PATH = "../data/processed/tickets_preprocessed.csv"
INDEX_NAME = "logistore_tickets"

if os.path.exists(PREPROCESSED_PATH):
    df_clean = pd.read_csv(PREPROCESSED_PATH)
    print(f"Dataset chargé — {df_clean.shape[0]} tickets, {df_clean.shape[1]} colonnes")
    print(f"   Colonnes : {list(df_clean.columns)}")
else:
    # Fallback sur le dataset brut si le prétraité n'existe pas encore
    df_clean = pd.read_csv("../data/raw/dataset-tickets-multi-lang3-4k.csv")
    print(f" Prétraité non trouvé — dataset brut chargé ({df_clean.shape[0]} tickets)")

# Vérification index OpenSearch
count = os_client.count(index=INDEX_NAME)
print(f"Index '{INDEX_NAME}' — {count['count']} documents indexés")

display(df_clean.head(3))

In [ ]:
# Import pipeline

from src.retrieval.hybrid_search import hybrid_search
from src.llm.llm_client import rag_answer, call_llm
from src.llm.rag_pipeline import run_rag, to_rag_tickets

In [ ]:
# Configuration pour l'évaluation

EVAL_MODEL = os.getenv("OPENROUTER_MODEL")  # nvidia/nemotron...
N_SAMPLES_PER_LANG = 10   # tickets échantillonnés par langue → 50 total
LANGUAGES = ["en", "de", "fr", "es", "pt"]
TOP_K = 5
EVAL_OUTPUT = Path("../data/eval/")
EVAL_OUTPUT.mkdir(parents=True, exist_ok=True)

## 1. Génération du jeu de test (LLM-as-judge)

In [ ]:
# Génération de requête

QUERY_GEN_PROMPT = """Voici un ticket de support client :

Sujet : {subject}
Problème : {body}

Génère UNE SEULE question courte (1-2 phrases max) qu'un utilisateur aurait 
pu poser pour décrire ce problème. La question doit être dans la langue du ticket 
({lang}), naturelle, sans mentionner de détails techniques spécifiques au ticket.
Réponds uniquement avec la question, sans explication."""

In [ ]:
def generate_query_for_ticket(row: dict) -> str:
    """Appelle le LLM pour générer une requête synthétique depuis un ticket.
    
    Args:
        row: Dict avec au moins 'subject', 'body', 'language'.
    
    Returns:
        Requête synthétique générée, ou None si échec.
    """
    lang_labels = {
        "en": "anglais", "fr": "français", "de": "allemand",
        "es": "espagnol", "pt": "portugais"
    }
    lang = str(row.get("language", "en")).lower()
    lang_label = lang_labels.get(lang, lang)

    # Texte source : subject + début du body
    subject = str(row.get("subject", "")).strip()
    body = str(row.get("body", row.get("text_clean", ""))).strip()[:300]

    prompt = QUERY_GEN_PROMPT.format(
        subject=subject,
        body=body,
        lang=lang_label,
    )

    try:
        messages = [{"role": "user", "content": prompt}]
        raw = call_llm(
            messages=messages,
            temperature=0.7,
            max_tokens=80,
        )
        # Nettoyage : strip, supprime guillemets, prend la première ligne seulement
        query = raw.strip().strip('"').strip("'").split("\n")[0].strip()
        return query if query else None

    except Exception as e:
        print(f"⚠️  Échec génération pour ticket {row.get('ticket_id', '?')} : {e}")
        return None


print("✅ Fonction generate_query_for_ticket définie")

In [ ]:
# --- Échantillonnage stratifié par langue ---
N_SAMPLES_PER_LANG = 10
LANGUAGES = ["en", "de", "fr", "es", "pt"]

# Normalise la colonne language en minuscules
df_clean["language"] = df_clean["language"].str.lower()

# Échantillonnage : 10 tickets par langue
df_sample = (
    df_clean[df_clean["language"].isin(LANGUAGES)]
    .groupby("language", group_keys=False)
    .apply(lambda g: g.sample(n=min(N_SAMPLES_PER_LANG, len(g)), random_state=42))
    .reset_index(drop=True)
)

# Ajoute ticket_id si absent (format ticket_XXXX comme dans OpenSearch)
if "ticket_id" not in df_sample.columns:
    df_sample["ticket_id"] = df_sample.index.map(lambda i: f"ticket_{i}")

print(f"✅ Échantillon : {len(df_sample)} tickets")
display(df_sample.groupby("language").size().rename("count").to_frame())

In [ ]:
import time

generated_queries = []

for i, (_, row) in enumerate(df_sample.iterrows()):
    print(f"[{i+1:02d}/{len(df_sample)}] langue={row['language']} | ticket={row['ticket_id']}", end=" → ")
    
    query = generate_query_for_ticket(row.to_dict())
    
    generated_queries.append({
        "ticket_id":       row["ticket_id"],
        "language":        row["language"],
        "type":            row.get("type", ""),
        "priority":        row.get("priority", ""),
        "subject":         row.get("subject", ""),
        "text":            str(row.get("text_clean", row.get("body", "")))[:300],
        "query_generated": query,
    })
    
    print(f'"{query[:60]}..."' if query else "❌ échec")
    
    # Sauvegarde intermédiaire tous les 10 tickets
    if (i + 1) % 10 == 0:
        pd.DataFrame(generated_queries).to_csv(
            "../data/eval/test_queries_partial.csv", index=False
        )
        print(f"   💾 Sauvegarde intermédiaire ({i+1} tickets)")
    
    time.sleep(0.5)  # rate limiting

df_eval = pd.DataFrame(generated_queries)
# Supprime les lignes où la génération a échoué
n_failed = df_eval["query_generated"].isna().sum()
print(f"\n✅ Génération terminée — {len(df_eval) - n_failed}/{len(df_eval)} requêtes générées ({n_failed} échecs)")

In [ ]:
import os
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(), override=True)

key = os.getenv("OPENROUTER_API_KEY", "")


In [ ]:
from importlib import reload
import src.llm.llm_client as llm_mod
reload(llm_mod)
from src.llm.llm_client import call_llm, rag_answer
print("✅ llm_client rechargé")

# Test rapide
import httpx, json
headers = {
    "Authorization": f"Bearer {os.getenv('OPENROUTER_API_KEY')}",
    "Content-Type": "application/json",
    "HTTP-Referer": "https://logistore-rag.local",
    "X-Title": "LogiStore RAG",
}
with httpx.Client(timeout=15) as client:
    resp = client.post(
        "https://openrouter.ai/api/v1/chat/completions",
        headers=headers,
        json={"model": "nvidia/nemotron-3-super-120b-a12b:free",
              "messages": [{"role": "user", "content": "Dis bonjour"}],
              "max_tokens": 10}
    )
    print(resp.status_code, resp.json())

In [ ]:
# --- Sauvegarde finale ---
EVAL_OUTPUT.mkdir(parents=True, exist_ok=True)
output_path = EVAL_OUTPUT / "test_queries.csv"

df_eval.to_csv(output_path, index=False)
print(f"✅ Jeu de test sauvegardé → {output_path}")
print(f"   {len(df_eval)} requêtes | {df_eval['query_generated'].notna().sum()} valides")

# --- Aperçu ---
display(
    df_eval[["ticket_id", "language", "type", "query_generated"]]
    .dropna(subset=["query_generated"])
    .head(10)
)

# --- Statistiques par langue ---
stats = df_eval.groupby("language").agg(
    total=("ticket_id", "count"),
    valides=("query_generated", lambda x: x.notna().sum()),
).assign(taux=lambda df: (df["valides"] / df["total"] * 100).round(1))

print("\nTaux de succès par langue :")
display(stats)

In [ ]:
load_dotenv(find_dotenv(), override=True)
from importlib import reload
import src.llm.llm_client as llm_mod
reload(llm_mod)
from src.llm.llm_client import call_llm